In [1]:
path = '/kaggle/input/datasets/almutalem/rehab24-6/REHAB24-6'

In [2]:
!ls -l {path}

total 68
drwxr-xr-x 8 nobody nogroup     0 Sep 15 04:22 2d_joints
drwxr-xr-x 8 nobody nogroup     0 Sep 15 04:22 2d_markers
drwxr-xr-x 8 nobody nogroup     0 Sep 15 04:22 3d_joints
drwxr-xr-x 8 nobody nogroup     0 Sep 15 04:22 3d_markers
-rw-r--r-- 1 nobody nogroup   395 Sep 15 04:22 joints_names.txt
-rw-r--r-- 1 nobody nogroup   581 Sep 15 04:22 marker_names.txt
-rw-r--r-- 1 nobody nogroup 53290 Sep 15 04:22 Segmentation.csv
-rw-r--r-- 1 nobody nogroup  1729 Sep 15 04:22 Segmentation.txt


In [3]:
import pandas as pd

seg_df = pd.read_csv(f'{path}/Segmentation.csv', sep=';')
seg_df.head()

,video_id,repetition_number,exercise_id,person_id,first_frame,last_frame,cam17_orientation,mocap_erroneous,exercise_subtype,lights_on,extra_person_in_cam17,extra_person_in_cam18,correctness
0,PM_000,1,1,1,180,377,front,0,right arm,0,3,0,1
1,PM_000,2,1,1,378,620,front,0,right arm,0,3,0,1
2,PM_000,3,1,1,621,865,front,0,right arm,0,3,0,1
3,PM_000,4,1,1,866,1085,front,0,right arm,0,3,3,1
4,PM_000,5,1,1,1086,1265,front,0,right arm,0,3,3,1


In [4]:
cols = ['extra_person_in_cam17', 'extra_person_in_cam18', 'correctness', 'mocap_erroneous']
{col: seg_df[col].value_counts().to_dict() for col in cols}

{'extra_person_in_cam17': {0: 686, 2: 206, 1: 117, 3: 63},
 'extra_person_in_cam18': {0: 1030, 1: 34, 3: 4, 2: 4},
 'correctness': {1: 568, 0: 504},
 'mocap_erroneous': {0: 1057, 1: 15}}

In [5]:
seg_df['exercise_id'].value_counts()

exercise_id
4    210
2    208
6    195
1    178
5    174
3    107
Name: count, dtype: int64

In [6]:
for col in cols: display(pd.crosstab(index=[seg_df['exercise_id']], columns=seg_df[col], dropna=False))

extra_person_in_cam17,0,1,2,3
exercise_id,,,,
1,127,23,22,6
2,139,23,42,4
3,57,14,18,18
4,116,10,53,31
5,117,17,38,2
6,130,30,33,2


extra_person_in_cam18,0,1,2,3
exercise_id,,,,
1,174,0,1,3
2,207,0,0,1
3,102,3,2,0
4,197,13,0,0
5,174,0,0,0
6,176,18,1,0


correctness,0,1
exercise_id,,
1,88,90
2,114,94
3,55,52
4,90,120
5,96,78
6,61,134


mocap_erroneous,0,1
exercise_id,,
1,178,0
2,208,0
3,96,11
4,210,0
5,174,0
6,191,4


- Will be using Camera 17 (fornt view) for exercise (1, 2, 3, 4) and Camera 18 (Side view) for the rest.
- Since the `mocap_errorenous` is only **17** we will remove it entierly.
- We will be also removing the row having `extra_person_in_cam17` and `extra_person_in_cam18`.
- Despite `extra_person_in_cam17` flag 3 value having **63** only **59** effective data will reduce since we are using cam17 in exercise 1, 2, 3, 4 and most dropiut is from exercise 4 **31**.
- Similarly in `extra_person_in_cam18` having **4** no impact will be in our data as the exercise containing it does not require cam18 for training.
- Similary for the falg 2 for cam17 net **135** instead of **206** and for cam18 net reduction drop is just **1**.

### Dropping the rows containing mocap error and extra flag 2 and 3.

In [7]:
clean_df = seg_df[(seg_df['mocap_erroneous'] == 0) & (seg_df['extra_person_in_cam17'] < 2) & (seg_df['extra_person_in_cam18'] < 2)]

In [8]:
clean_df.shape

(789, 13)

In [9]:
summary_table = clean_df.groupby('exercise_id')['correctness'].agg(
    total_count='count',
    correct_count='sum',
    wrong_count=lambda x: (x == 0).sum()
).astype(int)

display(summary_table)

,total_count,correct_count,wrong_count
exercise_id,,,
1,149,80,69
2,162,71,91
3,63,33,30
4,126,77,49
5,134,60,74
6,155,104,51


### Create a seperate npy per repeition per video

In [10]:
output_dir = './processed_repetitions'
features_to_extract = ['2d_joints', '2d_markers']

In [11]:
import os
import numpy as np
from tqdm.auto import tqdm

In [12]:
for feat in features_to_extract:
    for ex_id in range(1, 7): # Exercises 1 through 6
        os.makedirs(os.path.join(output_dir, feat, f'Ex{ex_id}'), exist_ok=True)

In [13]:
for index, row in tqdm(clean_df.iterrows(), total=len(clean_df), desc="Extracting Sequences"):
    vid = row['video_id']
    ex_id = row['exercise_id']
    rep_num = row['repetition_number']
    start_f = row['first_frame']
    end_f = row['last_frame']
    label = row['correctness']

    cam = 'c17' if ex_id in [1, 2, 3, 4] else 'c18'

    for feat in features_to_extract:
        src_file = f'{path}/{feat}/Ex{ex_id}/{vid}-{cam}-30fps.npy'

        if os.path.exists(src_file):
            full_array = np.load(src_file)
            rep_slice = full_array[start_f : end_f + 1]
            dest_file = f'{output_dir}/{feat}/Ex{ex_id}/{vid}_rep{rep_num}_label{label}.npy'
            np.save(dest_file, rep_slice)

Extracting Sequences:   0%|          | 0/789 [00:00<?, ?it/s]

In [14]:
!ls -l '/kaggle/working/processed_repetitions/2d_joints/Ex1/PM_001_rep10_label0.npy'

total 9496
-rw-r--r-- 1 root root  89568 Sep 15 04:24 PM_001_rep10_label0.npy
-rw-r--r-- 1 root root  81248 Sep 15 04:24 PM_001_rep1_label1.npy
-rw-r--r-- 1 root root  88736 Sep 15 04:24 PM_001_rep2_label1.npy
-rw-r--r-- 1 root root  80832 Sep 15 04:24 PM_001_rep3_label1.npy
-rw-r--r-- 1 root root  78336 Sep 15 04:24 PM_001_rep4_label1.npy
-rw-r--r-- 1 root root  71264 Sep 15 04:24 PM_001_rep5_label1.npy
-rw-r--r-- 1 root root 104544 Sep 15 04:24 PM_001_rep6_label0.npy
-rw-r--r-- 1 root root 119520 Sep 15 04:24 PM_001_rep7_label0.npy
-rw-r--r-- 1 root root  86656 Sep 15 04:24 PM_001_rep8_label0.npy
-rw-r--r-- 1 root root  80000 Sep 15 04:24 PM_001_rep9_label0.npy
-rw-r--r-- 1 root root  88320 Sep 15 04:24 PM_002_rep1_label1.npy
-rw-r--r-- 1 root root  87488 Sep 15 04:24 PM_002_rep2_label1.npy
-rw-r--r-- 1 root root  80416 Sep 15 04:24 PM_002_rep3_label1.npy
-rw-r--r-- 1 root root  60032 Sep 15 04:24 PM_002_rep4_label1.npy
-rw-r--r-- 1 root root  82496 Sep 15 04:24 PM_002_rep7_label0.np